# ISY503 Assessment 3 – NLP Sentiment Analysis Project

Student Name: Justin Komala Putra, Trevor Jordan Zeleang Kau, Marcus Chung Seng Lee  
Project Type: Natural Language Processing  
Dataset: Amazon Product Reviews  
Model: Bidirectional LSTM Neural Network  
Output: Positive review or Negative review

In [1]:
!pip install tensorflow pandas numpy scikit-learn -q

import os
import re
import tarfile
import urllib.request
import pickle
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.layers import TextVectorization


In [2]:
DATA_URL = "https://www.cs.jhu.edu/~mdredze/datasets/sentiment/processed_acl.tar.gz"
DATA_FILE = "processed_acl.tar.gz"
DATA_DIR = "processed_acl"
CATEGORY = "electronics"

if not os.path.exists(DATA_FILE):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

if not os.path.exists(DATA_DIR):
    print("Extracting dataset...")
    with tarfile.open(DATA_FILE, "r:gz") as tar:
        tar.extractall()

positive_path = os.path.join(DATA_DIR, CATEGORY, "positive.review")
negative_path = os.path.join(DATA_DIR, CATEGORY, "negative.review")

print("Positive path exists:", os.path.exists(positive_path))
print("Negative path exists:", os.path.exists(negative_path))
print("Dataset ready.")

Extracting dataset...


/tmp/ipykernel_7583/522856572.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Positive path exists: True
Negative path exists: True
Dataset ready.


In [3]:
def clean_word(word):
    return re.sub(r"[^a-zA-Z]", "", word.lower())


def parse_processed_review(line):
    words = []

    for token in line.split():
        if ":" in token:
            word, count = token.rsplit(":", 1)

            if word.startswith("#label#"):
                continue

            word = clean_word(word)

            if len(word) > 1:
                try:
                    count = int(float(count))
                except:
                    count = 1

                words.extend([word] * min(count, 3))

    return " ".join(words)


def load_reviews(category):
    positive_path = os.path.join(DATA_DIR, category, "positive.review")
    negative_path = os.path.join(DATA_DIR, category, "negative.review")

    reviews = []
    labels = []

    with open(positive_path, "r", encoding="latin-1") as file:
        for line in file:
            text = parse_processed_review(line)
            if len(text.split()) > 5:
                reviews.append(text)
                labels.append(1)

    with open(negative_path, "r", encoding="latin-1") as file:
        for line in file:
            text = parse_processed_review(line)
            if len(text.split()) > 5:
                reviews.append(text)
                labels.append(0)

    data = pd.DataFrame({
        "review": reviews,
        "label": labels
    })

    data = data.sample(frac=1, random_state=42).reset_index(drop=True)
    return data


data = load_reviews(CATEGORY)

print(data.head())
print("\nLabel Count:")
print(data["label"].value_counts())
print("\nTotal Reviews:", len(data))


                                              review  label
0  experiencewith setupand thefirmware upwhenyou ...      0
1  pc justa recommendyou blackfuzzy simpleeasy pa...      1
2  middle shipping shipping lessonlearned beep pa...      0
3  setup only my mask one tonum mentionedthat typ...      1
4  barely adf adf adf doesnot unusualdocuments th...      0

Label Count:
label
0    1000
1    1000
Name: count, dtype: int64

Total Reviews: 2000


In [4]:
X = data["review"].values
y = data["label"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (1400,)
Validation: (300,)
Testing: (300,)


In [5]:
MAX_WORDS = 10000

vectorizer = TextVectorization(
    max_tokens=MAX_WORDS,
    output_mode="tf_idf"
)

vectorizer.adapt(X_train)

X_train_vec = vectorizer(X_train)
X_val_vec = vectorizer(X_val)
X_test_vec = vectorizer(X_test)

In [6]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(MAX_WORDS,)),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │     1,280,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,288,449 (4.92 MB)

 Trainable params: 1,288,449 (4.92 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_vec,
    y_train,
    validation_data=(X_val_vec, y_val),
    epochs=20,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 7s 60ms/step - accuracy: 0.5786 - loss: 1.2907 - val_accuracy: 0.8400 - val_loss: 0.4870
Epoch 2/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.8514 - loss: 0.4064 - val_accuracy: 0.8000 - val_loss: 0.5147
Epoch 3/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.9336 - loss: 0.2335 - val_accuracy: 0.8533 - val_loss: 0.3732
Epoch 4/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.9729 - loss: 0.1120 - val_accuracy: 0.8567 - val_loss: 0.4165
Epoch 5/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.9757 - loss: 0.0921 - val_accuracy: 0.8333 - val_loss: 0.3803
Epoch 6/20
44/44 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step - accuracy: 0.9807 - loss: 0.0784 - val_accuracy: 0.8333 - val_loss: 0.4623


In [8]:
test_loss, test_accuracy = model.evaluate(X_test_vec, y_test)

print("Test Loss:", round(test_loss, 4))
print("Test Accuracy:", round(test_accuracy * 100, 2), "%")

y_pred_prob = model.predict(X_test_vec)
y_pred = (y_pred_prob >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Negative review", "Positive review"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8767 - loss: 0.3510
Test Loss: 0.351
Test Accuracy: 87.67 %
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

Classification Report:
                 precision    recall  f1-score   support

Negative review       0.86      0.91      0.88       150
Positive review       0.90      0.85      0.87       150

       accuracy                           0.88       300
      macro avg       0.88      0.88      0.88       300
   weighted avg       0.88      0.88      0.88       300


Confusion Matrix:
[[136  14]
 [ 23 127]]


In [9]:
def predict_sentiment(review):
    review_vector = vectorizer(np.array([review]))
    prediction_score = model.predict(review_vector, verbose=0)[0][0]

    if prediction_score >= 0.5:
        prediction_label = "Positive review"
    else:
        prediction_label = "Negative review"

    return prediction_label, prediction_score


In [10]:
sample_reviews = [
    "This product is excellent and works perfectly.",
    "Amazing quality and very useful.",
    "The item is easy to use and worth the money.",
    "Terrible product, completely broken and useless.",
    "Worst purchase ever, very disappointed and angry.",
    "It stopped working after one day and I regret buying it.",
    "Poor quality, waste of money and not recommended."
]

print("\n=== Model Prediction Results ===\n")

for i, review in enumerate(sample_reviews, 1):
    label, score = predict_sentiment(review)

    print(f"Example {i}")
    print("Review:", review)
    print("Predicted Sentiment:", label)
    print("Confidence Score:", round(float(score), 4))
    print("-" * 60)



=== Model Prediction Results ===

Example 1
Review: This product is excellent and works perfectly.
Predicted Sentiment: Positive review
Confidence Score: 0.5904
------------------------------------------------------------
Example 2
Review: Amazing quality and very useful.
Predicted Sentiment: Positive review
Confidence Score: 0.6157
------------------------------------------------------------
Example 3
Review: The item is easy to use and worth the money.
Predicted Sentiment: Positive review
Confidence Score: 0.5157
------------------------------------------------------------
Example 4
Review: Terrible product, completely broken and useless.
Predicted Sentiment: Negative review
Confidence Score: 0.3691
------------------------------------------------------------
Example 5
Review: Worst purchase ever, very disappointed and angry.
Predicted Sentiment: Negative review
Confidence Score: 0.4352
------------------------------------------------------------
Example 6
Review: It stopped working

In [11]:
test_review = "This product is really good and worth buying"

label, score = predict_sentiment(test_review)

print("Review:", test_review)
print("Sentiment:", label)
print("Confidence:", round(float(score), 4))

Review: This product is really good and worth buying
Sentiment: Positive review
Confidence: 0.5504


In [12]:
import pickle

model.save("sentiment_model.keras")

with open("vectorizer_data.pkl", "wb") as f:
    pickle.dump({
        "vocab": vectorizer.get_vocabulary(),
        "weights": vectorizer.get_weights()
    }, f)

print("Saved correctly")

Saved correctly


In [13]:
!rm vectorizer_vocab.pkl
!rm vectorizer.pkl

rm: cannot remove 'vectorizer_vocab.pkl': No such file or directory
rm: cannot remove 'vectorizer.pkl': No such file or directory


In [14]:
!ls

processed_acl	      sample_data	     vectorizer_data.pkl
processed_acl.tar.gz  sentiment_model.keras


In [15]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 89.7 MB/s eta 0:00:00


In [16]:
%%writefile app.py
import streamlit as st
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

st.title("Sentiment Analysis App")

st.write("This app predicts whether a product review is positive or negative.")

st.subheader("Instructions")
st.write("Enter at least 5 words and click Analyze.")

model = tf.keras.models.load_model("sentiment_model.keras")

with open("vectorizer_data.pkl", "rb") as f:
    vectorizer_data = pickle.load(f)

raw_vocab = vectorizer_data["vocab"]

# remove reserved tokens and duplicates
vocab = []
seen = set()
for word in raw_vocab:
    word = str(word)
    if word in ["", "[UNK]"]:
        continue
    if word not in seen:
        vocab.append(word)
        seen.add(word)

vectorizer = TextVectorization(
    max_tokens=10000,
    output_mode="multi_hot"
)

vectorizer.set_vocabulary(vocab)

user_input = st.text_area("Enter a review:")

if st.button("Analyze Sentiment"):
    if len(user_input.split()) < 5:
        st.warning("Enter at least 5 words")
    else:
        review_vector = vectorizer(tf.constant([user_input]))
        score = model.predict(review_vector, verbose=0)[0][0]

        st.write(f"Confidence: {round(float(score), 4)}")

        if score >= 0.5:
            st.success("Positive review")
        else:
            st.error("Negative review")

Writing app.py


In [17]:
!pkill -f streamlit

In [18]:
!streamlit run app.py &



2026-05-09 02:53:38.717 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.229.126.47:8501

  Stopping...


In [19]:
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token("3DAYjeaubA2KHjnwhV9iVu3Ba2g_2hdafThNVqgCRFurBdcKD")

public_url = ngrok.connect(8501, "http")
print(public_url.public_url)

https://acid-railway-spinner.ngrok-free.dev
